# Data Ingestion

In [1]:
import logging
import os
from contextlib import contextmanager

import duckdb

# Configure structured logging (force=True so re-running this cell in an
# already-running kernel doesn't just add duplicate handlers).
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(name)s: %(message)s",
    force=True,
)
logger = logging.getLogger("EcoLens.DataPipeline")

DEFAULT_DB_PATH = (
    "/Users/macbook/Project/personal/EcoLens/services/data-pipeline"
    "/data/historical/ecolens_historical.duckdb"
)
db_path = os.getenv("DUCKDB_PATH", DEFAULT_DB_PATH)


@contextmanager
def duckdb_connection(path: str, read_only: bool = True):
    """Connect, log, and guarantee the close+log happens exactly once --
    defined here so every later cell does `with duckdb_connection(...) as con:`
    instead of re-pasting this same try/except/finally block (which is how
    the notebook ended up with two copies of the same ~40 lines, and the
    same "DuckDB connection closed cleanly." log line, drifting out of
    sync with each other after only one of them got edited).
    """
    resolved_path = os.path.expanduser(path)
    if read_only and not os.path.exists(resolved_path):
        logger.error("DuckDB database file not found at path: %s", resolved_path)
        raise FileNotFoundError(f"Database file not found: {resolved_path}")

    con = None
    try:
        con = duckdb.connect(database=resolved_path, read_only=read_only)
        logger.info(
            "Successfully connected (read_only=%s) to %s", read_only, resolved_path
        )
        yield con
    except duckdb.Error as e:
        logger.exception("DuckDB internal error while connecting to %s", resolved_path)
        raise ConnectionError(f"Failed to connect to DuckDB database: {e}") from e
    finally:
        if con is not None:
            con.close()
            logger.info("DuckDB connection closed cleanly.")


with duckdb_connection(db_path) as con:
    version = con.execute("SELECT version()").fetchone()[0]
    logger.info("DuckDB Version active: %s", version)

    tables_df = con.execute(
        "SELECT table_name FROM duckdb_tables WHERE internal = false"
    ).df()
    print("Tables in database:")
    for name in tables_df["table_name"]:
        print(f" - {name}")


2026-07-27 10:28:14,891 [INFO] EcoLens.DataPipeline: Successfully connected (read_only=True) to /Users/macbook/Project/personal/EcoLens/services/data-pipeline/data/historical/ecolens_historical.duckdb
2026-07-27 10:28:14,897 [INFO] EcoLens.DataPipeline: DuckDB Version active: v1.5.5
2026-07-27 10:28:15,640 [INFO] EcoLens.DataPipeline: DuckDB connection closed cleanly.


Tables in database:
 - aemo_holidays
 - aemo_nem_dispatch
 - aemo_wem_dispatch
 - bom_observations
 - openelectricity_responses


In [2]:
"""
Per-source data-quality audit -- one check per raw table this pipeline
ingests from (aemo_nem_dispatch, aemo_wem_dispatch,
openelectricity_responses, bom_observations, aemo_holidays), so every
source PreProcessing/Feature selection/Feature Engineering below
eventually draws from has actually been looked at, not just assumed
complete. Shared "continuous daily timeline vs actual record count"
logic factored into one function -- the four dispatch/weather sources
are all real per-day time series, so they share the same check; only
the EXPECTED_PER_DAY constant and which table/timestamp column differs
per source. aemo_holidays isn't a per-day time series (see its own
cell below for why it gets a different check).
"""

from datetime import datetime, timezone

import pandas as pd


def daily_record_counts(con, table_name: str, ts_expr: str, start: str, end: str) -> pd.DataFrame:
    """Continuous daily timeline from `start` to `end` (exclusive), actual
    row count per day -- 0, not a missing row, for a day with none at all.
    """
    query = f"""
        WITH date_spine AS (
            SELECT UNNEST(generate_series(
                CAST(? AS DATE), CAST(? AS DATE) - INTERVAL 1 DAY, INTERVAL 1 DAY
            ))::date AS data_date
        ),
        aggregated AS (
            SELECT CAST({ts_expr} AS DATE) AS data_date, COUNT(*) AS record_count
            FROM {table_name}
            WHERE {ts_expr} >= ? AND {ts_expr} < ?
            GROUP BY CAST({ts_expr} AS DATE)
        )
        SELECT ds.data_date, COALESCE(a.record_count, 0) AS record_count
        FROM date_spine ds
        LEFT JOIN aggregated a ON a.data_date = ds.data_date
        ORDER BY ds.data_date
    """
    return con.execute(query, [start, end, start, end]).df()


def summarize_gaps(df: pd.DataFrame, expected_per_day: int, label: str) -> pd.DataFrame:
    """Prints a one-line summary and returns the short/missing days (empty
    if none) -- callers display that directly rather than the whole
    day-by-day table.
    """
    missing_days = int((df["record_count"] == 0).sum())
    short_days = int(((df["record_count"] > 0) & (df["record_count"] < expected_per_day)).sum())
    print(
        f"{label}: {len(df)} days in range, expected {expected_per_day} records/day\n"
        f"  {missing_days} day(s) with zero records, {short_days} day(s) short (partial)"
    )
    return df[df["record_count"] < expected_per_day]


In [3]:
"""
aemo_nem_dispatch -- AEMO NEM 5-min dispatch data: NSW1/QLD1/VIC1/SA1/
TAS1 per-region market rows plus one network-level "NEM" row per `ts`
carrying the fuel-tech mix (see int_energy_unified_30min's header for
why that split matters downstream). 6 distinct `region` values total,
confirmed against this table directly -- EXPECTED_PER_DAY below is
288 records/day (24h * 60min / 5min) * 6, not a guessed round number.
"""
START = "2025-08-01 00:00:00"
END = datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M:%S")
EXPECTED_PER_DAY = 288 * 6  # 5-min grain * {NSW1, QLD1, VIC1, SA1, TAS1, NEM}

with duckdb_connection(db_path) as con:
    nem_daily = daily_record_counts(con, "aemo_nem_dispatch", "ts", START, END)

short_days = summarize_gaps(nem_daily, EXPECTED_PER_DAY, "aemo_nem_dispatch")
short_days


2026-07-27 10:28:15,690 [INFO] EcoLens.DataPipeline: Successfully connected (read_only=True) to /Users/macbook/Project/personal/EcoLens/services/data-pipeline/data/historical/ecolens_historical.duckdb
2026-07-27 10:28:15,772 [INFO] EcoLens.DataPipeline: DuckDB connection closed cleanly.


aemo_nem_dispatch: 360 days in range, expected 1728 records/day
  1 day(s) with zero records, 2 day(s) short (partial)


,data_date,record_count
0,2025-08-01,1002
358,2026-07-25,726
359,2026-07-26,0


In [4]:
"""
aemo_wem_dispatch -- AEMO WEM (Western Australia) dispatch data, single
`region` = "WEM". Documented elsewhere in this repo as a 30-minute
series (48 records/day) -- checked against the real data instead of
trusting that: the actual ingested rows land at 288/day, the same
5-minute cadence as NEM. Real finding, not a bug in this check -- worth
a doc fix wherever "AEMO WEM 30-minute dispatch" is written, out of
scope for this notebook to fix on its own.
"""
START = "2025-08-01 00:00:00"
END = datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M:%S")
EXPECTED_PER_DAY = 288  # empirically 5-min grain, not the documented 30-min/48-per-day

with duckdb_connection(db_path) as con:
    wem_daily = daily_record_counts(con, "aemo_wem_dispatch", "ts", START, END)

short_days = summarize_gaps(wem_daily, EXPECTED_PER_DAY, "aemo_wem_dispatch")
short_days


2026-07-27 10:28:15,803 [INFO] EcoLens.DataPipeline: Successfully connected (read_only=True) to /Users/macbook/Project/personal/EcoLens/services/data-pipeline/data/historical/ecolens_historical.duckdb
2026-07-27 10:28:15,828 [INFO] EcoLens.DataPipeline: DuckDB connection closed cleanly.


aemo_wem_dispatch: 360 days in range, expected 288 records/day
  3 day(s) with zero records, 8 day(s) short (partial)


,data_date,record_count
0,2025-08-01,216
82,2025-10-22,72
83,2025-10-23,0
84,2025-10-24,0
85,2025-10-25,216
91,2025-10-31,72
92,2025-11-01,216
93,2025-11-02,72
94,2025-11-03,216
358,2026-07-25,72


In [5]:
"""
openelectricity_responses -- OpenElectricity (OpenNEM) network-level
fallback data, keyed by `network_code`/`region` = "NEM" or "WEM" (2
distinct values, confirmed against this table directly) -- not
per-state, see int_energy_unified_30min's header on why it can only
substitute for NEM's broadcast fuel-tech mix, never a missing
per-region market row. `ts` is stored as text (ISO-8601 with offset),
cast to TIMESTAMPTZ before bucketing -- same cast PreProcessing's
oe_30min CTE uses.
"""
START = "2025-08-01 00:00:00"
END = datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M:%S")
EXPECTED_PER_DAY = 288 * 2  # 5-min grain * {NEM, WEM}

with duckdb_connection(db_path) as con:
    oe_daily = daily_record_counts(
        con, "openelectricity_responses", "CAST(ts AS TIMESTAMPTZ)", START, END
    )

short_days = summarize_gaps(oe_daily, EXPECTED_PER_DAY, "openelectricity_responses")
short_days


2026-07-27 10:28:15,858 [INFO] EcoLens.DataPipeline: Successfully connected (read_only=True) to /Users/macbook/Project/personal/EcoLens/services/data-pipeline/data/historical/ecolens_historical.duckdb
2026-07-27 10:28:15,905 [INFO] EcoLens.DataPipeline: DuckDB connection closed cleanly.


openelectricity_responses: 360 days in range, expected 576 records/day
  0 day(s) with zero records, 1 day(s) short (partial)


,data_date,record_count
359,2026-07-26,524


In [6]:
"""
bom_observations -- BoM weather observations (or Open-Meteo ERA5
historical backfill using the same schema), one row per station per
30-min slot across 6 regions (NSW1/QLD1/VIC1/SA1/TAS1/WEM, confirmed
against this table directly) -- already on the 30-min grain
PreProcessing needs, no resampling required.
"""
START = "2025-08-01 00:00:00"
END = datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M:%S")
EXPECTED_PER_DAY = 48 * 6  # 30-min grain * {NSW1, QLD1, VIC1, SA1, TAS1, WEM}

with duckdb_connection(db_path) as con:
    bom_daily = daily_record_counts(con, "bom_observations", "ts", START, END)

short_days = summarize_gaps(bom_daily, EXPECTED_PER_DAY, "bom_observations")
short_days


2026-07-27 10:28:15,931 [INFO] EcoLens.DataPipeline: Successfully connected (read_only=True) to /Users/macbook/Project/personal/EcoLens/services/data-pipeline/data/historical/ecolens_historical.duckdb
2026-07-27 10:28:15,957 [INFO] EcoLens.DataPipeline: DuckDB connection closed cleanly.


bom_observations: 360 days in range, expected 288 records/day
  0 day(s) with zero records, 1 day(s) short (partial)


,data_date,record_count
0,2025-08-01,216


In [7]:
"""
aemo_holidays -- public holiday calendar per region, NOT a per-day
time series (a handful of dates a year, not one row per 30-min slot),
so the daily-timeline gap check above doesn't apply -- the real
question for a reference table like this is date coverage: which
years/regions actually have rows.

This directly quantifies the gap PreProcessing's holiday_flags CTE
already calls out in a comment ("aemo_holidays currently only has 2026
rows"): if every region's min(date) is 2026-01-01, every 2025 slot in
feature_table_30min gets is_public_holiday = FALSE, not "unknown" --
a real, currently-uncorrected gap, not something this audit cell can
fix on its own.
"""
with duckdb_connection(db_path) as con:
    holiday_coverage = con.execute(
        """
        select region, count(*) as n_holidays, min(date) as first_date, max(date) as last_date
        from aemo_holidays
        group by region
        order by region
        """
    ).df()

print("aemo_holidays coverage by region:")
print(holiday_coverage.to_string(index=False))

missing_2025 = (holiday_coverage["first_date"] > "2025-12-31").all()
print(
    f"\nevery region's first recorded holiday is in 2026 or later: {missing_2025}"
    + ("  -> 2025 holiday flags in feature_table_30min are all FALSE, not unknown"
       if missing_2025 else "")
)


2026-07-27 10:28:15,990 [INFO] EcoLens.DataPipeline: Successfully connected (read_only=True) to /Users/macbook/Project/personal/EcoLens/services/data-pipeline/data/historical/ecolens_historical.duckdb
2026-07-27 10:28:16,009 [INFO] EcoLens.DataPipeline: DuckDB connection closed cleanly.


aemo_holidays coverage by region:
region  n_holidays first_date  last_date
  NSW1          14 2026-01-01 2026-12-28
  QLD1          11 2026-01-01 2026-12-26
   SA1          14 2026-01-01 2026-12-28
  TAS1          12 2026-01-01 2026-12-26
  VIC1          15 2026-01-01 2026-12-28
   WEM          14 2026-01-01 2026-12-28

every region's first recorded holiday is in 2026 or later: True  -> 2025 holiday flags in feature_table_30min are all FALSE, not unknown


# Warehousing

## Data preprocessing

In [8]:
from datetime import datetime, timezone

# Master timeline start per the spec's Temporal Spine table; END defaults
# to "now" so re-running this cell later just picks up whatever's landed
# in the meantime.
START = "2025-08-01 00:00:00"
END = datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M:%S")

REGIONS = ["NSW1", "QLD1", "VIC1", "SA1", "TAS1", "WEM"]

# Market data (demand/price/interconnector flow) -- present per-region
# for NEM sub-regions, and on WEM's own single-zone rows.
MARKET_COLS = [
    "demand_mw", "price_mwh",
    "interconnector_imports_mw", "interconnector_exports_mw", "net_import_mw",
]
# Fuel-tech generation mix -- present on NEM's network-level 'NEM' row
# (broadcast to all 5 sub-regions below) and on WEM's own rows.
FUELTECH_COLS = [
    "coal_black_mw", "coal_brown_mw", "gas_ccgt_mw", "gas_ocgt_mw", "gas_other_mw",
    "hydro_mw", "wind_mw", "solar_utility_mw", "solar_rooftop_mw", "biomass_mw",
    "pumped_hydro_mw", "distillate_mw", "battery_discharge_mw", "battery_charge_mw",
    "total_generation_mw", "renewable_proportion", "emissions_intensity_kgco2e_per_mwh",
]

# Averaging 5-min MW readings into one 30-min figure is the standard way
# to downsample power (not energy) data -- every column here is an
# instantaneous MW rate, not a cumulative MWh total, so AVG is correct
# for all of them (the spec's "total for energy" case doesn't apply,
# since none of our raw columns are actual energy/MWh quantities).
def _avg_cols(cols: list[str]) -> str:
    return ",\n        ".join(f"AVG({c}) AS {c}" for c in cols)


def _coalesce_market(cols: list[str]) -> str:
    # AEMO first; OpenElectricity only fills a gap for WEM (direct
    # region match) -- it reports NEM at the whole-network level only,
    # never per state, so it can't substitute for a missing NSW1/QLD1/
    # VIC1/SA1/TAS1 market row at this granularity.
    return ",\n    ".join(f"COALESCE(m.{c}, oe_wem.{c}) AS {c}" for c in cols)


def _coalesce_fueltech(cols: list[str]) -> str:
    # AEMO first (WEM's own row, or NEM's broadcast network-level row);
    # OpenElectricity fallback works for both here since fuel mix is
    # inherently network-wide on both sides.
    return ",\n    ".join(
        f"COALESCE(CASE WHEN s.region = 'WEM' THEN w.{c} ELSE f.{c} END, oe.{c}) AS {c}"
        for c in cols
    )


query = f"""
WITH regions AS (
    SELECT * FROM (VALUES {", ".join(f"('{r}')" for r in REGIONS)}) AS t(region)
),
spine AS (
    -- One row per (region, 30-min slot) across the whole master timeline --
    -- everything else LEFT JOINs onto this so a missing source never
    -- collapses a slot, it just leaves that slot's columns null.
    SELECT r.region, ts_slot
    FROM regions r
    CROSS JOIN (
        SELECT UNNEST(generate_series(
            CAST(? AS TIMESTAMP), CAST(? AS TIMESTAMP) - INTERVAL 30 MINUTE, INTERVAL 30 MINUTE
        )) AS ts_slot
    ) t
),
nem_market_30min AS (
    SELECT region, time_bucket(INTERVAL 30 MINUTE, ts) AS ts_slot,
        {_avg_cols(MARKET_COLS)},
        ANY_VALUE(data_quality_status) AS data_quality_status
    FROM aemo_nem_dispatch
    WHERE region != 'NEM' AND ts >= ? AND ts < ?
    GROUP BY region, time_bucket(INTERVAL 30 MINUTE, ts)
),
nem_fueltech_30min AS (
    SELECT time_bucket(INTERVAL 30 MINUTE, ts) AS ts_slot,
        {_avg_cols(FUELTECH_COLS)}
    FROM aemo_nem_dispatch
    WHERE region = 'NEM' AND ts >= ? AND ts < ?
    GROUP BY time_bucket(INTERVAL 30 MINUTE, ts)
),
wem_30min AS (
    SELECT region, time_bucket(INTERVAL 30 MINUTE, ts) AS ts_slot,
        {_avg_cols(MARKET_COLS)},
        {_avg_cols(FUELTECH_COLS)},
        ANY_VALUE(data_quality_status) AS data_quality_status
    FROM aemo_wem_dispatch
    WHERE ts >= ? AND ts < ?
    GROUP BY region, time_bucket(INTERVAL 30 MINUTE, ts)
),
-- OpenElectricity's `ts` is stored as text (ISO-8601 with offset), not
-- a native timestamp -- cast before bucketing/filtering.
oe_30min AS (
    SELECT region, time_bucket(INTERVAL 30 MINUTE, CAST(ts AS TIMESTAMPTZ)) AS ts_slot,
        {_avg_cols(MARKET_COLS)},
        {_avg_cols(FUELTECH_COLS)}
    FROM openelectricity_responses
    WHERE CAST(ts AS TIMESTAMPTZ) >= ? AND CAST(ts AS TIMESTAMPTZ) < ?
    GROUP BY region, time_bucket(INTERVAL 30 MINUTE, CAST(ts AS TIMESTAMPTZ))
),
-- BoM is already on the 30-min grain -- map straight across, no
-- resampling needed (GROUP BY is just a defensive dedupe).
weather_30min AS (
    SELECT region, ts AS ts_slot,
        ANY_VALUE(temp_c) AS temp_c, ANY_VALUE(apparent_temp_c) AS apparent_temp_c,
        ANY_VALUE(dew_point_c) AS dew_point_c, ANY_VALUE(humidity_pct) AS humidity_pct,
        ANY_VALUE(wind_speed_kmh) AS wind_speed_kmh, ANY_VALUE(wind_direction_deg) AS wind_direction_deg,
        ANY_VALUE(wind_gust_kmh) AS wind_gust_kmh, ANY_VALUE(pressure_hpa) AS pressure_hpa,
        ANY_VALUE(rain_since_9am_mm) AS rain_since_9am_mm, ANY_VALUE(cloud_cover_pct) AS cloud_cover_pct
    FROM bom_observations
    WHERE ts >= ? AND ts < ?
    GROUP BY region, ts
),
-- Daily flag, broadcast onto every 30-min slot that date via the join
-- below (not resampled -- holidays don't have a time component).
-- Note: aemo_holidays currently only has 2026 rows, so this broadcasts
-- FALSE for the back half of 2025 rather than a real "unknown" --
-- known gap, not something this query can fix on its own.
holiday_flags AS (
    SELECT region, CAST(date AS DATE) AS holiday_date, TRUE AS is_public_holiday
    FROM aemo_holidays
)
SELECT
    s.region,
    s.ts_slot AS ts,
    {_coalesce_market(MARKET_COLS)},
    {_coalesce_fueltech(FUELTECH_COLS)},
    CASE
        WHEN s.region = 'WEM' AND w.ts_slot IS NOT NULL THEN 'aemo_wem'
        WHEN s.region != 'WEM' AND m.ts_slot IS NOT NULL THEN 'aemo_nem'
        WHEN oe_wem.ts_slot IS NOT NULL OR oe.ts_slot IS NOT NULL THEN 'openelectricity'
        ELSE NULL
    END AS market_source,
    wx.temp_c, wx.apparent_temp_c, wx.dew_point_c, wx.humidity_pct,
    wx.wind_speed_kmh, wx.wind_direction_deg, wx.wind_gust_kmh,
    wx.pressure_hpa, wx.rain_since_9am_mm, wx.cloud_cover_pct,
    COALESCE(h.is_public_holiday, FALSE) AS is_public_holiday
FROM spine s
LEFT JOIN nem_market_30min m ON m.region = s.region AND m.ts_slot = s.ts_slot
LEFT JOIN nem_fueltech_30min f ON f.ts_slot = s.ts_slot AND s.region != 'WEM'
LEFT JOIN wem_30min w ON w.region = s.region AND w.ts_slot = s.ts_slot
LEFT JOIN oe_30min oe ON oe.region = 'NEM' AND oe.ts_slot = s.ts_slot AND s.region != 'WEM'
LEFT JOIN oe_30min oe_wem ON oe_wem.region = 'WEM' AND oe_wem.ts_slot = s.ts_slot AND s.region = 'WEM'
LEFT JOIN weather_30min wx ON wx.region = s.region AND wx.ts_slot = s.ts_slot
LEFT JOIN holiday_flags h ON h.region = s.region AND h.holiday_date = CAST(s.ts_slot AS DATE)
ORDER BY s.region, s.ts_slot;
"""
# 6 (start, end) pairs above: spine, nem_market, nem_fueltech, wem,
# oe, weather -- each needs its own copy of START/END.
query_params = [START, END] * 6


with duckdb_connection(db_path) as con:
    feature_table_30min = con.execute(query, query_params).df()

print(f"shape: {feature_table_30min.shape}")
print("\nrows per region:")
print(feature_table_30min["region"].value_counts())
print("\nmarket_source breakdown (how each row's demand/price/fuel-mix was sourced):")
print(feature_table_30min["market_source"].value_counts(dropna=False))
print("\nnull % per column:")
print(feature_table_30min.isna().mean().round(3))

feature_table_30min.head(10)

2026-07-27 10:28:16,055 [INFO] EcoLens.DataPipeline: Successfully connected (read_only=True) to /Users/macbook/Project/personal/EcoLens/services/data-pipeline/data/historical/ecolens_historical.duckdb
2026-07-27 10:29:25,121 [INFO] EcoLens.DataPipeline: DuckDB connection closed cleanly.


shape: (103728, 36)

rows per region:
region
NSW1    17288
QLD1    17288
SA1     17288
TAS1    17288
VIC1    17288
WEM     17288
Name: count, dtype: int64

market_source breakdown (how each row's demand/price/fuel-mix was sourced):
market_source
aemo_nem           85925
aemo_wem           16944
openelectricity      779
NaN                   80
Name: count, dtype: int64

null % per column:
region                                0.000
ts                                    0.000
demand_mw                             0.005
price_mwh                             0.005
interconnector_imports_mw             0.172
interconnector_exports_mw             0.172
net_import_mw                         0.005
coal_black_mw                         0.004
coal_brown_mw                         0.167
gas_ccgt_mw                           0.004
gas_ocgt_mw                           0.004
gas_other_mw                          0.167
hydro_mw                              0.167
wind_mw                             

,region,ts,demand_mw,price_mwh,interconnector_imports_mw,interconnector_exports_mw,net_import_mw,coal_black_mw,coal_brown_mw,gas_ccgt_mw,...,apparent_temp_c,dew_point_c,humidity_pct,wind_speed_kmh,wind_direction_deg,wind_gust_kmh,pressure_hpa,rain_since_9am_mm,cloud_cover_pct,is_public_holiday
0,NSW1,2025-08-01 00:00:00,NaN,NaN,NaN,NaN,NaN,11807.998683,4812.583400,1168.959867,...,NaN,NaN,NaN,NaN,<NA>,NaN,NaN,NaN,<NA>,False
1,NSW1,2025-08-01 00:30:00,NaN,NaN,NaN,NaN,NaN,11792.260300,4816.276083,1251.472383,...,NaN,NaN,NaN,NaN,<NA>,NaN,NaN,NaN,<NA>,False
2,NSW1,2025-08-01 01:00:00,NaN,NaN,NaN,NaN,NaN,12012.461033,4824.953183,1403.057950,...,NaN,NaN,NaN,NaN,<NA>,NaN,NaN,NaN,<NA>,False
3,NSW1,2025-08-01 01:30:00,NaN,NaN,NaN,NaN,NaN,12397.095250,4818.552133,1516.505233,...,NaN,NaN,NaN,NaN,<NA>,NaN,NaN,NaN,<NA>,False
4,NSW1,2025-08-01 02:00:00,NaN,NaN,NaN,NaN,NaN,12820.257533,4819.281300,1513.242633,...,NaN,NaN,NaN,NaN,<NA>,NaN,NaN,NaN,<NA>,False
5,NSW1,2025-08-01 02:30:00,NaN,NaN,NaN,NaN,NaN,12860.189300,4829.166733,1644.700367,...,NaN,NaN,NaN,NaN,<NA>,NaN,NaN,NaN,<NA>,False
6,NSW1,2025-08-01 03:00:00,NaN,NaN,NaN,NaN,NaN,12867.516350,4844.989600,1681.701533,...,NaN,NaN,NaN,NaN,<NA>,NaN,NaN,NaN,<NA>,False
7,NSW1,2025-08-01 03:30:00,NaN,NaN,NaN,NaN,NaN,12896.135133,4845.906300,1680.447450,...,NaN,NaN,NaN,NaN,<NA>,NaN,NaN,NaN,<NA>,False
8,NSW1,2025-08-01 04:00:00,NaN,NaN,NaN,NaN,NaN,12811.594350,4848.802133,1560.432983,...,NaN,NaN,NaN,NaN,<NA>,NaN,NaN,NaN,<NA>,False
9,NSW1,2025-08-01 04:30:00,NaN,NaN,NaN,NaN,NaN,12747.176200,4855.484433,1557.037733,...,NaN,NaN,NaN,NaN,<NA>,NaN,NaN,NaN,<NA>,False


# # Feature selection 

In [9]:
"""
Feature selection -- the 5-step statistical gauntlet from TODO.md /
HYBRID_ML_PIPELINE_AND_MODEL_TRAINING_SPEC.md, run against
`feature_table_30min` (built in the PreProcessing section above).

Step 4 uses LightGBM's built-in gain importance instead of true
TreeSHAP: the `shap` package fails to build on this machine (Python
3.12, no compatible wheel/sdist available at the time of writing) --
swap in `shap.TreeExplainer` here if that gets resolved later, the
rest of the pipeline doesn't care which importance measure feeds it.
Step 5 (TFT variable-selection gating) can't run at all yet -- there's
no TFT in this repo to gate with (see TODO.md's "Predictive model"
section) -- see that step's cell below for what's blocking it.
"""

import numpy as np
import pandas as pd

df = feature_table_30min.copy()

TARGET = "demand_mw"
IDENTIFIER_COLS = ["region", "ts"]
# market_source is provenance we added ourselves during preprocessing
# (which source filled each row) -- useful for auditing, not a real
# predictive feature, so it's excluded from every step below.
METADATA_COLS = ["market_source"]

candidate_cols = [
    c for c in df.columns if c not in IDENTIFIER_COLS + METADATA_COLS + [TARGET]
]

# ── Step 1: Structural hygiene ──────────────────────────────────────
# Zero-variance columns: Australia has no nuclear/geothermal generation
# so those columns never existed in our schema to begin with -- this
# check is the general-purpose version of that rule, so it also catches
# anything else that turns out to be constant.
numeric_candidates = df[candidate_cols].select_dtypes(include=[np.number, bool]).columns.tolist()
variances = df[numeric_candidates].var(numeric_only=True)
zero_variance_cols = variances[variances.fillna(0) == 0].index.tolist()

# Fully-null columns: a whole feature with zero real observations
# anywhere -- distinct from the per-row nulls WEM/NEM's structural
# split naturally produces (those are real "this fuel type doesn't
# exist in this grid" signal, not missing data, see PreProcessing).
null_pct = df[candidate_cols].isna().mean().sort_values(ascending=False)
fully_null_cols = null_pct[null_pct == 1.0].index.tolist()

structural_drop_cols = zero_variance_cols + fully_null_cols

print("Step 1 -- structural hygiene")
print(f"  zero-variance columns: {zero_variance_cols or '(none)'}")
print(f"  fully-null columns: {fully_null_cols or '(none)'}")
print(f"  highest null% columns:\n{null_pct.head(8)}")
print(f"  -> dropping {len(structural_drop_cols)} columns: {structural_drop_cols}")


Step 1 -- structural hygiene
  zero-variance columns: (none)
  fully-null columns: (none)
  highest null% columns:
interconnector_exports_mw             0.171632
interconnector_imports_mw             0.171632
pumped_hydro_mw                       0.167342
emissions_intensity_kgco2e_per_mwh    0.167342
coal_brown_mw                         0.167342
gas_other_mw                          0.167342
hydro_mw                              0.167342
solar_rooftop_mw                      0.167342
dtype: float64
  -> dropping 0 columns: []


In [10]:
# ── Step 2: Information-Theoretic Ranking (Mutual Information) ──────
# MI catches non-linear relationships plain correlation would miss
# (e.g. price spikes driving demand response). Needs a NaN-free matrix
# and a numeric target -- median-impute just for this ranking (this
# doesn't touch feature_table_30min itself), and drop rows with no
# target at all rather than imputing the thing we're trying to predict.
from sklearn.feature_selection import mutual_info_regression

mi_input_cols = [c for c in candidate_cols if c not in structural_drop_cols]

mi_df = df.dropna(subset=[TARGET]).copy()
mi_df["is_public_holiday"] = mi_df["is_public_holiday"].astype(int)

X_mi = mi_df[mi_input_cols].apply(lambda s: s.fillna(s.median()))
y_mi = mi_df[TARGET]

mi_scores = mutual_info_regression(X_mi, y_mi, random_state=42)
mi_series = pd.Series(mi_scores, index=mi_input_cols).sort_values(ascending=False)

DROP_FRACTION = 0.175  # midpoint of the spec's "bottom 15-20%"
n_drop = int(len(mi_series) * DROP_FRACTION)
low_mi_cols = mi_series.tail(n_drop).index.tolist()

print("Step 2 -- Mutual Information ranking against demand_mw")
print(mi_series.to_string())
print(f"\n  -> dropping bottom {DROP_FRACTION:.0%} ({n_drop} columns): {low_mi_cols}")


Step 2 -- Mutual Information ranking against demand_mw
total_generation_mw                   0.624532
net_import_mw                         0.500458
coal_black_mw                         0.482544
interconnector_imports_mw             0.432270
interconnector_exports_mw             0.423422
coal_brown_mw                         0.396941
hydro_mw                              0.384119
solar_rooftop_mw                      0.353418
pumped_hydro_mw                       0.341557
emissions_intensity_kgco2e_per_mwh    0.314829
biomass_mw                            0.302047
gas_ocgt_mw                           0.298500
renewable_proportion                  0.282069
gas_other_mw                          0.281332
gas_ccgt_mw                           0.281300
battery_charge_mw                     0.276588
price_mwh                             0.270561
battery_discharge_mw                  0.185421
dew_point_c                           0.173119
wind_direction_deg                    0.164502
appar

In [11]:
# ── Step 3: Time-Series Dependency (PACF) ───────────────────────────
# Validates the spec's proposed lag steps (1, 2, 48, 336 = 30min/1hr/
# 1day/1week) against demand_mw's actual Partial Autocorrelation,
# rather than trusting round numbers. Run on one representative region
# (NSW1, the largest/most complete series) -- PACF needs a single
# continuous series, not a multi-region panel. ffill/bfill just for
# this check (small gaps only, see PreProcessing's null% diagnostics --
# this doesn't touch feature_table_30min).
from statsmodels.tsa.stattools import pacf

PACF_REGION = "NSW1"
MAX_LAG = 340  # a little past the widest proposed lag (336)

nsw1_demand = (
    df[df["region"] == PACF_REGION]
    .sort_values("ts")[TARGET]
    .ffill()
    .bfill()
)

pacf_vals, confint = pacf(nsw1_demand, nlags=MAX_LAG, alpha=0.05, method="ywm")
ci_half_width = confint[:, 1] - pacf_vals
significant_lags = np.where(np.abs(pacf_vals) > ci_half_width)[0]
significant_lags = significant_lags[significant_lags > 0]  # lag 0 is trivially 1.0

proposed_lags = [1, 2, 48, 336]

print(f"Step 3 -- PACF on {PACF_REGION} demand_mw (up to lag {MAX_LAG})")
print(f"  {len(significant_lags)} statistically significant lags found")
print("  checking the spec's proposed lags against that:")
for lag in proposed_lags:
    verdict = "significant" if lag in significant_lags else "NOT significant"
    print(f"    lag {lag:>3} (pacf={pacf_vals[lag]:+.4f}): {verdict}")


Step 3 -- PACF on NSW1 demand_mw (up to lag 340)
  131 statistically significant lags found
  checking the spec's proposed lags against that:
    lag   1 (pacf=+0.9838): significant
    lag   2 (pacf=-0.8629): significant
    lag  48 (pacf=-0.2158): significant
    lag 336 (pacf=-0.0647): significant


In [12]:
# ── Step 4: Multicollinearity Pruning (LightGBM importance + corr) ──
# True TreeSHAP needs the `shap` package (see this section's intro
# cell for why it's not installed here) -- LightGBM's own gain-based
# feature_importances_ is the practical substitute: fit once, use it
# both as the "which feature wins" tiebreaker below and as a sanity
# check on Step 2's MI ranking.
import lightgbm as lgb

model = lgb.LGBMRegressor(n_estimators=200, max_depth=6, random_state=42, verbosity=-1)
model.fit(X_mi, y_mi)  # same NaN-free matrix Step 2 built
importance = pd.Series(model.feature_importances_, index=mi_input_cols).sort_values(
    ascending=False
)

print("Step 4a -- LightGBM gain importance (TreeSHAP substitute)")
print(importance.to_string())

# Multicollinearity: any pair of candidate features correlated above
# 0.9 is redundant -- keep whichever one LightGBM actually leans on
# more, drop the other. This is where "drop redundant macro aggregates
# while retaining granular fuel types" (e.g. total_generation_mw vs.
# the per-fuel breakdown) actually gets decided, rather than assumed.
corr_matrix = X_mi.corr().abs()
upper_triangle = corr_matrix.where(
    np.triu(np.ones(corr_matrix.shape), k=1).astype(bool)
)

CORR_THRESHOLD = 0.9
multicollinear_pairs = [
    (col, other, upper_triangle.loc[other, col])
    for col in upper_triangle.columns
    for other in upper_triangle.index[upper_triangle[col] > CORR_THRESHOLD]
]

multicollinear_drop_cols: set[str] = set()
print(f"\nStep 4b -- multicollinearity pairs (|r| > {CORR_THRESHOLD})")
for a, b, r in sorted(multicollinear_pairs, key=lambda p: -p[2]):
    loser = a if importance[a] < importance[b] else b
    multicollinear_drop_cols.add(loser)
    print(f"  {a} <-> {b}  r={r:.3f}  -> drop {loser} (lower LightGBM importance)")

print(f"\n  -> dropping {len(multicollinear_drop_cols)} columns: {multicollinear_drop_cols}")


Step 4a -- LightGBM gain importance (TreeSHAP substitute)
wind_direction_deg                    530
pressure_hpa                          447
dew_point_c                           433
interconnector_exports_mw             429
price_mwh                             358
interconnector_imports_mw             274
coal_brown_mw                         247
wind_speed_kmh                        224
solar_rooftop_mw                      218
humidity_pct                          210
coal_black_mw                         209
temp_c                                197
net_import_mw                         175
cloud_cover_pct                       170
biomass_mw                            163
apparent_temp_c                       156
total_generation_mw                   149
distillate_mw                         145
wind_mw                               140
wind_gust_kmh                         139
pumped_hydro_mw                       139
gas_other_mw                          137
emissions_intensit

In [13]:
# ── Step 5: Automated TFT Gating ─────────────────────────────────────
# Blocked, not skipped: this step needs an actual trained TFT to read
# Variable Selection Network weights from, and there's no TFT anywhere
# in this repo yet -- "Stand up the TFT" is still an open item in
# TODO.md's "Predictive model" section. Recording that dependency
# explicitly here (rather than silently omitting the step) so it's
# obvious what unblocks it and this cell can be filled in later without
# re-deriving why it was empty.
print(
    "Step 5 -- TFT variable-selection gating: BLOCKED\n"
    "  Needs a trained TFT (TODO.md 'Predictive model' -> 'Stand up the TFT')\n"
    "  to read Variable Selection Network attention weights from.\n"
    "  Once that exists: feed it the columns surviving Steps 1-4 below,\n"
    "  run inference across a representative set of forecast horizons,\n"
    "  and permanently drop whatever gets near-zero VSN weight everywhere."
)


Step 5 -- TFT variable-selection gating: BLOCKED
  Needs a trained TFT (TODO.md 'Predictive model' -> 'Stand up the TFT')
  to read Variable Selection Network attention weights from.
  Once that exists: feed it the columns surviving Steps 1-4 below,
  run inference across a representative set of forecast horizons,
  and permanently drop whatever gets near-zero VSN weight everywhere.


In [14]:
# ── Apply the decisions from Steps 1, 2 and 4 (Step 3 informs Feature
# Engineering's lag choice, it doesn't drop columns; Step 5 is blocked) ──
all_drop_cols = sorted(set(structural_drop_cols) | set(low_mi_cols) | multicollinear_drop_cols)
kept_cols = [c for c in candidate_cols if c not in all_drop_cols]

selected_feature_table_30min = df[IDENTIFIER_COLS + [TARGET] + kept_cols].copy()

print(f"Feature selection summary: {len(candidate_cols)} candidate columns -> {len(kept_cols)} kept")
print(f"  Step 1 (structural hygiene):    {sorted(structural_drop_cols)}")
print(f"  Step 2 (bottom {DROP_FRACTION:.0%} MI):        {sorted(low_mi_cols)}")
print(f"  Step 4 (multicollinearity):     {sorted(multicollinear_drop_cols)}")
print(f"\nkept: {kept_cols}")

selected_feature_table_30min.head()


Feature selection summary: 32 candidate columns -> 25 kept
  Step 1 (structural hygiene):    []
  Step 2 (bottom 18% MI):        ['cloud_cover_pct', 'is_public_holiday', 'rain_since_9am_mm', 'wind_gust_kmh', 'wind_speed_kmh']
  Step 4 (multicollinearity):     ['apparent_temp_c', 'total_generation_mw', 'wind_gust_kmh']

kept: ['price_mwh', 'interconnector_imports_mw', 'interconnector_exports_mw', 'net_import_mw', 'coal_black_mw', 'coal_brown_mw', 'gas_ccgt_mw', 'gas_ocgt_mw', 'gas_other_mw', 'hydro_mw', 'wind_mw', 'solar_utility_mw', 'solar_rooftop_mw', 'biomass_mw', 'pumped_hydro_mw', 'distillate_mw', 'battery_discharge_mw', 'battery_charge_mw', 'renewable_proportion', 'emissions_intensity_kgco2e_per_mwh', 'temp_c', 'dew_point_c', 'humidity_pct', 'wind_direction_deg', 'pressure_hpa']


,region,ts,demand_mw,price_mwh,interconnector_imports_mw,interconnector_exports_mw,net_import_mw,coal_black_mw,coal_brown_mw,gas_ccgt_mw,...,distillate_mw,battery_discharge_mw,battery_charge_mw,renewable_proportion,emissions_intensity_kgco2e_per_mwh,temp_c,dew_point_c,humidity_pct,wind_direction_deg,pressure_hpa
0,NSW1,2025-08-01 00:00:00,NaN,NaN,NaN,NaN,NaN,11807.998683,4812.583400,1168.959867,...,0.000000,100.114750,662.884300,17.100000,1429.429967,NaN,NaN,NaN,<NA>,NaN
1,NSW1,2025-08-01 00:30:00,NaN,NaN,NaN,NaN,NaN,11792.260300,4816.276083,1251.472383,...,-0.016667,47.780017,721.527017,17.968333,1432.140717,NaN,NaN,NaN,<NA>,NaN
2,NSW1,2025-08-01 01:00:00,NaN,NaN,NaN,NaN,NaN,12012.461033,4824.953183,1403.057950,...,-0.016667,14.026633,474.015033,17.955000,1453.720983,NaN,NaN,NaN,<NA>,NaN
3,NSW1,2025-08-01 01:30:00,NaN,NaN,NaN,NaN,NaN,12397.095250,4818.552133,1516.505233,...,-0.033333,20.272633,242.493517,18.130000,1486.123700,NaN,NaN,NaN,<NA>,NaN
4,NSW1,2025-08-01 02:00:00,NaN,NaN,NaN,NaN,NaN,12820.257533,4819.281300,1513.242633,...,-0.083333,265.048650,138.589950,19.456667,1537.315483,NaN,NaN,NaN,<NA>,NaN


## # Feature Engineering

In [15]:
"""
Feature engineering -- TODO.md's "Feature Engineering" section /
HYBRID_ML_PIPELINE_AND_MODEL_TRAINING_SPEC.md's "Generation &
Engineering" step. Builds demand_mw lag + rolling-window features on
top of `selected_feature_table_30min` (the Feature selection section
above), using exactly the 4 lags Step 3's PACF run validated as
statistically significant (1, 2, 48, 336 -- 30min/1hr/1day/1week at
the 30-min grain) rather than the dense 1-48 lag set the production
`ml_features_demand_v1` dbt mart uses -- this notebook is the smaller,
PACF-justified feature set the spec describes; see that mart's own
header comment for the denser alternative already running in the
warehouse.

Critical correctness requirement: each of the 6 regions is its own
independent time series -- a lag or rolling window computed without
grouping by region would leak QLD1's early rows into NSW1's late ones
wherever the sort happens to butt them up against each other. Every
computation below groups by `region` first, verified explicitly.
"""

import numpy as np
import pandas as pd

LAG_STEPS = [1, 2, 48, 336]  # PACF-validated in Feature selection Step 3
ROLLING_WINDOW = 336  # 7 days * 48 slots/day, same as dbt's ml_rolling_window_slots var
ROLLING_STATS = ["mean", "median", "max", "min", "std"]

engineered = selected_feature_table_30min.sort_values(["region", "ts"]).reset_index(
    drop=True
)

# ── Lag features ─────────────────────────────────────────────────────
demand_by_region = engineered.groupby("region", sort=False)[TARGET]
for lag in LAG_STEPS:
    engineered[f"demand_lag_{lag}"] = demand_by_region.shift(lag)

lag_cols = [f"demand_lag_{lag}" for lag in LAG_STEPS]
print("Lag features built:", lag_cols)
print("null % (boundary rows at the start of each region's history, expected):")
print(engineered[lag_cols].isna().mean())

# Correctness check, not just a comment: the first row of every region
# except the very first (alphabetically, NSW1) must have a NaN lag_1 --
# if any of them don't, a lag leaked across a region boundary.
first_rows = engineered.groupby("region", sort=False).head(1)
leaked = first_rows[first_rows["demand_lag_1"].notna()]
assert leaked.empty, f"Lag leaked across a region boundary:\n{leaked[['region', 'ts']]}"
print("\nregion-boundary check passed: no region's first row has a non-null lag_1")


Lag features built: ['demand_lag_1', 'demand_lag_2', 'demand_lag_48', 'demand_lag_336']
null % (boundary rows at the start of each region's history, expected):
demand_lag_1      0.005100
demand_lag_2      0.005100
demand_lag_48     0.005428
demand_lag_336    0.020400
dtype: float64

region-boundary check passed: no region's first row has a non-null lag_1


In [16]:
# ── Rolling-window stats ─────────────────────────────────────────────
# Trailing window, excluding the current row (shift(1) first) -- same
# convention ml_features_demand_v1.sql uses (`rows between 336
# preceding and 1 preceding`), so the model never sees a feature that
# peeks at the value it's trying to predict.
shifted_demand = demand_by_region.shift(1)
rolling = shifted_demand.groupby(engineered["region"]).rolling(
    ROLLING_WINDOW, min_periods=1
)
for stat in ROLLING_STATS:
    engineered[f"demand_rolling_{stat}_7d"] = getattr(rolling, stat)().reset_index(
        level=0, drop=True
    )

rolling_cols = [f"demand_rolling_{stat}_7d" for stat in ROLLING_STATS]
print("Rolling features built:", rolling_cols)
print("null %:")
print(engineered[rolling_cols].isna().mean())

# Correctness check: for one region, one arbitrary row, the rolling
# mean must equal a manual mean over that exact trailing window -- not
# just "did it run", did it compute the right window.
check_region = "NSW1"
check_idx = 500
region_df = engineered[engineered["region"] == check_region].reset_index(drop=True)
manual_mean = region_df.loc[check_idx - ROLLING_WINDOW : check_idx - 1, TARGET].mean()
computed_mean = region_df.loc[check_idx, "demand_rolling_mean_7d"]
assert np.isclose(manual_mean, computed_mean), (
    f"rolling mean mismatch: manual={manual_mean}, computed={computed_mean}"
)
print(
    f"\nspot-check passed ({check_region} row {check_idx}): "
    f"manual={manual_mean:.3f} == computed={computed_mean:.3f}"
)


Rolling features built: ['demand_rolling_mean_7d', 'demand_rolling_median_7d', 'demand_rolling_max_7d', 'demand_rolling_min_7d', 'demand_rolling_std_7d']
null %:
demand_rolling_mean_7d      0.001022
demand_rolling_median_7d    0.001022
demand_rolling_max_7d       0.001022
demand_rolling_min_7d       0.001022
demand_rolling_std_7d       0.001080
dtype: float64

spot-check passed (NSW1 row 500): manual=8418.200 == computed=8418.200


In [17]:
# ── Drop the raw, un-engineered column now that its lag/rolling
# features exist ────────────────────────────────────────────────────
# "Raw" here means demand_mw itself as a model INPUT -- keeping it in
# X alongside its own lags would be leakage (it's literally the value
# being predicted), not a redundancy concern like Feature selection's
# Step 4. `kept_cols` already excludes demand_mw (Feature selection's
# `candidate_cols` excluded the target from the start), so it stays
# out of X here for free -- it still needs to exist as the separate
# target column y, and every other covariate that survived Feature
# selection (price, fuel mix, weather) is carried through unchanged.
engineered_feature_cols = lag_cols + rolling_cols + kept_cols

X_engineered = engineered[IDENTIFIER_COLS + engineered_feature_cols].copy()
y_engineered = engineered[IDENTIFIER_COLS + [TARGET]].copy()

print("Feature engineering summary:")
print("  demand_mw itself: kept only as the target (y), excluded from features (X)")
print(f"  lag features:      {lag_cols}")
print(f"  rolling features:  {rolling_cols}")
print(f"  other covariates:  {kept_cols}")
print(f"\n  X_engineered: {X_engineered.shape}, y_engineered: {y_engineered.shape}")

X_engineered.head()


Feature engineering summary:
  demand_mw itself: kept only as the target (y), excluded from features (X)
  lag features:      ['demand_lag_1', 'demand_lag_2', 'demand_lag_48', 'demand_lag_336']
  rolling features:  ['demand_rolling_mean_7d', 'demand_rolling_median_7d', 'demand_rolling_max_7d', 'demand_rolling_min_7d', 'demand_rolling_std_7d']
  other covariates:  ['price_mwh', 'interconnector_imports_mw', 'interconnector_exports_mw', 'net_import_mw', 'coal_black_mw', 'coal_brown_mw', 'gas_ccgt_mw', 'gas_ocgt_mw', 'gas_other_mw', 'hydro_mw', 'wind_mw', 'solar_utility_mw', 'solar_rooftop_mw', 'biomass_mw', 'pumped_hydro_mw', 'distillate_mw', 'battery_discharge_mw', 'battery_charge_mw', 'renewable_proportion', 'emissions_intensity_kgco2e_per_mwh', 'temp_c', 'dew_point_c', 'humidity_pct', 'wind_direction_deg', 'pressure_hpa']

  X_engineered: (103728, 36), y_engineered: (103728, 3)


,region,ts,demand_lag_1,demand_lag_2,demand_lag_48,demand_lag_336,demand_rolling_mean_7d,demand_rolling_median_7d,demand_rolling_max_7d,demand_rolling_min_7d,...,distillate_mw,battery_discharge_mw,battery_charge_mw,renewable_proportion,emissions_intensity_kgco2e_per_mwh,temp_c,dew_point_c,humidity_pct,wind_direction_deg,pressure_hpa
0,NSW1,2025-08-01 00:00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.000000,100.114750,662.884300,17.100000,1429.429967,NaN,NaN,NaN,<NA>,NaN
1,NSW1,2025-08-01 00:30:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,-0.016667,47.780017,721.527017,17.968333,1432.140717,NaN,NaN,NaN,<NA>,NaN
2,NSW1,2025-08-01 01:00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,-0.016667,14.026633,474.015033,17.955000,1453.720983,NaN,NaN,NaN,<NA>,NaN
3,NSW1,2025-08-01 01:30:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,-0.033333,20.272633,242.493517,18.130000,1486.123700,NaN,NaN,NaN,<NA>,NaN
4,NSW1,2025-08-01 02:00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,-0.083333,265.048650,138.589950,19.456667,1537.315483,NaN,NaN,NaN,<NA>,NaN
